# Наложение масок на изображения

Этот ноутбук показывает три режима просмотра:
- размеченный `train` из `dl-lab-3-product-segmentation`
- псевдомаски для `unlabeled` (`350` изображений)
- псевдомаски для всего `dl-lab-1-image-classification` (`12392` изображений)

Для большого `lab1` ноутбук не пытается вывести все картинки сразу. Вместо этого есть удобные функции для:
- случайных примеров
- последовательных батчей по `start/count`
- просмотра overlay поверх исходного изображения

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["image.cmap"] = "gray"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def guess_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "dl-lab-3-product-segmentation").exists():
            return candidate
    raise FileNotFoundError("Project root with dl-lab-3-product-segmentation was not found.")


PROJECT_ROOT = guess_project_root()
SEG_DATASET_DIR = PROJECT_ROOT / "dl-lab-3-product-segmentation"
PSEUDO_ROOT = PROJECT_ROOT / "seg_runs" / "advanced_baseline" / "ensemble_outputs"

SOURCES = {
    "train_gt": {
        "title": "Labeled Train GT",
        "image_root": SEG_DATASET_DIR / "train" / "images",
        "mask_root": SEG_DATASET_DIR / "train" / "masks",
        "recursive": False,
    },
    "unlabeled_pseudo": {
        "title": "Unlabeled Pseudo Labels",
        "image_root": SEG_DATASET_DIR / "unlabeled" / "images",
        "mask_root": PSEUDO_ROOT / "unlabeled" / "masks",
        "recursive": False,
    },
    "lab1_pseudo": {
        "title": "Lab1 Recursive Pseudo Labels",
        "image_root": PROJECT_ROOT / "dl-lab-1-image-classification",
        "mask_root": PSEUDO_ROOT / "dl_lab_1_image_classification_all" / "masks",
        "recursive": True,
    },
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
for source_name, cfg in SOURCES.items():
    print(f"[{source_name}] image_root={cfg['image_root']} | exists={cfg['image_root'].exists()}")
    print(f"[{source_name}] mask_root ={cfg['mask_root']} | exists={cfg['mask_root'].exists()}")

In [ ]:
def collect_image_paths(root: Path) -> list:
    return sorted(
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )


def build_mask_path(image_path: Path, source_name: str) -> Path:
    cfg = SOURCES[source_name]
    if cfg["recursive"]:
        relative_path = image_path.relative_to(cfg["image_root"]).with_suffix(".png")
        return cfg["mask_root"] / relative_path
    return cfg["mask_root"] / image_path.with_suffix(".png").name


def load_image(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


def load_mask(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def overlay_mask(image: np.ndarray, mask: np.ndarray, color=(255, 0, 0), alpha=0.45) -> np.ndarray:
    image = image.astype(np.float32)
    result = image.copy()

    binary = mask > 0
    color_arr = np.array(color, dtype=np.float32)
    result[binary] = (1 - alpha) * result[binary] + alpha * color_arr

    return np.clip(result, 0, 255).astype(np.uint8)


def get_available_pairs(source_name: str) -> list:
    cfg = SOURCES[source_name]
    image_paths = collect_image_paths(cfg["image_root"])
    pairs = []
    for image_path in image_paths:
        mask_path = build_mask_path(image_path, source_name)
        if mask_path.exists():
            pairs.append((image_path, mask_path))
    return pairs


def summarize_source(source_name: str) -> dict:
    cfg = SOURCES[source_name]
    image_paths = collect_image_paths(cfg["image_root"])
    pairs = get_available_pairs(source_name)
    return {
        "source_name": source_name,
        "title": cfg["title"],
        "image_root": str(cfg["image_root"]),
        "mask_root": str(cfg["mask_root"]),
        "num_images": len(image_paths),
        "num_masks_found": len(pairs),
    }


def show_triplet(image: np.ndarray, mask: np.ndarray, overlay: np.ndarray, title: str = "") -> None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(image)
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title("Mask")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    if title:
        fig.suptitle(title)

    plt.tight_layout()
    plt.show()


def show_examples(source_name: str, count: int = 6, random_seed: int = 42, start: int = 0, shuffle: bool = True) -> None:
    pairs = get_available_pairs(source_name)
    if not pairs:
        print(f"No available image/mask pairs for source: {source_name}")
        return

    if shuffle:
        rng = random.Random(random_seed)
        pairs = pairs.copy()
        rng.shuffle(pairs)
        selected = pairs[:count]
    else:
        selected = pairs[start:start + count]

    if not selected:
        print(f"Empty selection for source={source_name}, start={start}, count={count}")
        return

    fig, axes = plt.subplots(len(selected), 3, figsize=(16, 4 * len(selected)))
    if len(selected) == 1:
        axes = np.array([axes])

    for row, (image_path, mask_path) in enumerate(selected):
        image = load_image(image_path)
        mask = load_mask(mask_path)
        overlay = overlay_mask(image, mask)
        rel_path = image_path.relative_to(SOURCES[source_name]["image_root"])

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(str(rel_path))
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask, cmap="gray", vmin=0, vmax=255)
        axes[row, 1].set_title(mask_path.name)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title("Overlay")
        axes[row, 2].axis("off")

    fig.suptitle(f"{SOURCES[source_name]['title']} | shown={len(selected)}")
    plt.tight_layout()
    plt.show()

In [ ]:
source_summaries = [summarize_source(name) for name in SOURCES]
for row in source_summaries:
    print(row)

## По одному примеру из каждого источника

Это быстрый sanity check, что пути и маски читаются правильно и overlay строится корректно.

In [ ]:
for source_name in SOURCES:
    pairs = get_available_pairs(source_name)
    if not pairs:
        print(f"[skip] {source_name}: no pairs found")
        continue

    image_path, mask_path = pairs[0]
    image = load_image(image_path)
    mask = load_mask(mask_path)
    overlay = overlay_mask(image, mask)

    print(f"Source: {source_name}")
    print(f"Image:  {image_path}")
    print(f"Mask:   {mask_path}")
    print(f"Shape:  {image.shape}")
    print(f"Unique mask values: {np.unique(mask)[:10]}")
    show_triplet(image, mask, overlay, title=SOURCES[source_name]["title"])

## Случайные примеры

Ниже сразу видно, как выглядят:
- ground truth на train
- псевдомаски на `350` unlabeled
- псевдомаски на всём `lab1`

In [ ]:
show_examples("train_gt", count=4, random_seed=42, shuffle=True)
show_examples("unlabeled_pseudo", count=6, random_seed=42, shuffle=True)
show_examples("lab1_pseudo", count=6, random_seed=42, shuffle=True)

## Батчи для большого `lab1`

Для `12392` изображений удобнее листать батчами, а не пытаться вывести всё разом.
Если нужно, просто меняйте `start`.

In [ ]:
LAB1_BATCH_START = 0
LAB1_BATCH_SIZE = 8
show_examples("lab1_pseudo", start=LAB1_BATCH_START, count=LAB1_BATCH_SIZE, shuffle=False)